<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Train, validation and test sets

**[Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/)** · Machine Learning: From Problem to Reliable Model · Module 3, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** explain why a model is judged on data it did not learn from, and to use the three sets (train, validation, test) and cross-validation for their own jobs. You see a model that memorises: perfect on the tickets it learned from, poor on new ones.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | The baselines and the rule "flag priority 1", from [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). Features and labels, from [Supervised learning](https://learning.nextia-ai.com/courses/ml/m01/supervised-learning/#features-and-labels). |
| **You do not need** | A local project. The setup below downloads the data. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M03-L01-train-validation-and-test-sets/train-validation-and-test-sets-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the practice data of this course (about 2.5 MB) into a folder `ticket-model/data`, and checks each file's checksum. It is the same data that `get_data.py` downloads in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 16939 rows (1983 escalated), valid 1179 (183), test 1002 (157).` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-model` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the C05 data into ticket-model/data/ and check it.
# Standard library only: the same steps as get_data.py.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/"
FILES = {
    "train.csv": "63b8732a28e891189cc28a9bcd505a093a9c4c51c01f3d0e1fba6d4959242d53",
    "valid.csv": "3604df83ea2ec3cec70c2eedfc7e95cc6d5cb82ccfc3b5cf565e614783e2bde2",
    "test.csv": "60e89f75e4d7d59ccfbda49ca719ce7f8e8d97c0380256d633ab240d546771ec",
    "extra_features.csv": "03c6da28bcb347982aa70a7c72b869771a6cb055654d629da924512c6fd56563",
    "daily_tickets.csv": "7deba76b112b4133a5d16b74be7e2b331ce28c7064e7fd8dd471768e5f1ba360",
    "july_tickets.csv": "10dffc093993341a361172d0aa6ea0e7c06521f140cc2a80ee90c713361f7c99",
    "july_labels.csv": "3163c57e398db6fd1c19e505548d0fa1464821ef96c77901a759abbd3a8605fa",
}
PROJECT_FILES = []
HERE = Path.cwd()
# The notebook works in its own folder ticket-model/, marked with a file, so
# it never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("ticket-model").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-model and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)
for name in PROJECT_FILES:
    fetch(name, LABS + "project/" + name, PROJECT / name)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
counts = {}
for part in ("train", "valid", "test"):
    rows = Path(f"data/{part}.csv").read_text(encoding="utf-8").splitlines()[1:]
    counts[part] = (len(rows), sum(row.endswith(",1") for row in rows))
print("Ready: train {} rows ({} escalated), valid {} ({}), test {} ({}).".format(
    *counts["train"], *counts["valid"], *counts["test"]))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the helpers of this lesson

The next cell defines the tools that this lesson uses. You do not need to read their code now.

- `read()` reads a ticket file, as in [Translate a need](https://learning.nextia-ai.com/courses/ml/m02/translate-a-need/). `FEATURES` lists the 10 feature columns from the SQL course, and `TARGET` is `escalated_72h`.
- `flag_report()` compares 0/1 flags with the real answers, as in [Define a baseline](https://learning.nextia-ai.com/courses/ml/m02/define-a-baseline/). It counts the escalations **caught** and **missed**, the **false alarms** and the tickets **correctly left** alone. Then it computes the share flagged, the precision, the recall and the accuracy.
- `memorising_tree()` makes a model that memorises: a **decision tree** with no limit. A decision tree asks yes/no questions about a ticket's features, one after another, until it reaches an answer. With no limit, it keeps asking until every group of training tickets has one answer. [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/) explains how it works.
- `id_rule()` makes a rule that flags about 1 ticket in 5 from the ticket's ID only. It is a rule that knows nothing about the ticket.

Run the cell. It prints nothing.

In [ ]:
# Helpers for this lesson.
import hashlib
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

CATEGORIES = ["channel", "team", "segment", "region"]
NUMBERS = ["priority", "order_value", "word_count", "customer_tenure_days", "prior_tickets_90d", "created_hour"]
FEATURES = CATEGORIES + NUMBERS
TARGET = "escalated_72h"

def read(path):
    """Read a ticket file. Only an empty cell is missing: "unknown" stays a category."""
    return pd.read_csv(path, parse_dates=["created_at"], dtype={"customer_id": "string"},
                       keep_default_na=False, na_values=[""])

def flag_report(y, flags):
    """Compare 0/1 flags with the real 0/1 answers: four counts and four shares."""
    y = np.asarray(y, dtype=bool)
    flags = np.asarray(flags, dtype=bool)
    caught, false_alarms = int((flags & y).sum()), int((flags & ~y).sum())
    missed, correctly_left = int((~flags & y).sum()), int((~flags & ~y).sum())
    return {"flagged": caught + false_alarms, "share flagged": round((caught + false_alarms) / len(y), 3),
            "caught": caught, "false alarms": false_alarms, "missed": missed, "correctly left": correctly_left,
            "precision": round(caught / (caught + false_alarms), 3) if caught + false_alarms else None,
            "recall": round(caught / (caught + missed), 3), "accuracy": round((caught + correctly_left) / len(y), 3)}

def memorising_tree():
    """A decision tree with no limit, and the preparation of the columns that it needs."""
    prepare = ColumnTransformer([
        ("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
        ("numbers", SimpleImputer(strategy="median", add_indicator=True), NUMBERS),
    ])
    return Pipeline([("prepare", prepare), ("model", DecisionTreeClassifier(random_state=0))])

def id_rule(number, part):
    """Rule `number`: flag about 1 ticket in 5, chosen from the ticket ID only."""
    return part["ticket_id"].map(lambda t: hashlib.sha256(f"{number}:{t}".encode()).digest()[0] < 51)

pd.set_option("display.width", 140)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Practice, chapter exercises and the final exam

A student learns mathematics from **practice exercises**, with the answers in the back of the book. To check the student's understanding, the teacher gives **end-of-chapter exercises**: new questions on the same topics. They show the weak points, and the student works on them. At the end, the **final exam** has new questions again. It is kept sealed until the exam, it is taken once, and its mark is the one that other people trust.

A model learns in the same way. It learns from the **training set** (the practice exercises). We compare and improve it on the **validation set** (the end-of-chapter exercises). We report one final score on the **test set** (the final exam). The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#three-sets-three-jobs) has a picture of the three sets.

Why so much care? Because a mark on the practice exercises tells you very little. A student who memorised the answers gets every practice question right, and still understands nothing. Section 2 shows this on 12 tickets.

## 2. Memorising and understanding, on 12 tickets

Run the next cell. It makes 8 practice tickets, with their answers, and 4 new tickets.

In [ ]:
practice = pd.DataFrame({
    "ticket": ["P-1", "P-2", "P-3", "P-4", "P-5", "P-6", "P-7", "P-8"],
    "priority": [1, 3, 2, 1, 3, 1, 2, 3],
    "team": ["payment", "delivery", "warranty", "returns", "account", "payment", "delivery", "returns"],
    "word_count": [120, 40, 210, 60, 25, 95, 70, 30],
    "escalated": [1, 0, 1, 0, 0, 1, 0, 0],
})
new = pd.DataFrame({
    "ticket": ["N-1", "N-2", "N-3", "N-4"],
    "priority": [1, 3, 2, 2],
    "team": ["payment", "delivery", "warranty", "returns"],
    "word_count": [110, 45, 190, 50],
    "escalated": [1, 0, 1, 0],
})
new

> **Check.** You should see four new tickets. N-1 and N-3 escalated. No new ticket is an exact copy of a practice ticket.

Two "models" learn from the practice tickets:

- The **memoriser** remembers the answer of every practice ticket. For a ticket, it looks for an exact copy (the same priority, team and word count) and gives the remembered answer. If it finds no copy, it says "no".
- The **rule** learned one pattern: priority 1 tickets escalate more often. It flags every priority 1 ticket.

> **Predict.** Which one is better on the practice tickets? Which one is better on the new tickets? Then run the cell.

In [ ]:
COLUMNS = ["priority", "team", "word_count"]
memory = {tuple(row[COLUMNS]): row["escalated"] for _, row in practice.iterrows()}

def memoriser(part):
    """The remembered answer for an exact copy of a practice ticket, else 0."""
    return [memory.get(tuple(row[COLUMNS]), 0) for _, row in part.iterrows()]

def rule(part):
    """Flag priority 1."""
    return (part["priority"] == 1).tolist()

tiny = pd.DataFrame.from_dict({
    "memoriser, practice": flag_report(practice["escalated"], memoriser(practice)),
    "rule, practice": flag_report(practice["escalated"], rule(practice)),
    "memoriser, new": flag_report(new["escalated"], memoriser(new)),
    "rule, new": flag_report(new["escalated"], rule(new)),
}, orient="index")
tiny[["flagged", "caught", "missed", "false alarms", "accuracy"]]

> **Check.** You should see that the memoriser is perfect on the practice tickets: it catches 3 of 3, with accuracy `1.0`. On the new tickets, it catches 0 of 2. The rule catches 2 of 3 practice tickets (accuracy `0.75`) and 1 of 2 new tickets.

The memoriser learned the tickets, not the pattern. On the practice tickets, it looks perfect. On new tickets, it has nothing to remember, so it knows nothing. The rule is less impressive on the practice tickets, and it keeps most of its skill on the new ones.

When a model learns the details of its training examples so well that it works much worse on new examples, it **overfits**. A score on the training data cannot show overfitting: there, the memoriser looks like the best model.

## 3. The real data: a model that memorises

Now the same test with a real model and the real tickets. `memorising_tree()` is a decision tree with no limit (see the helper cell). Run the next cell. It loads train and the validation set (May), and fits the tree on the 16,939 train tickets. It takes a few seconds.

In [ ]:
train = read("data/train.csv")
valid = read("data/valid.csv")
memoriser_tree = memorising_tree().fit(train[FEATURES], train[TARGET])
print("Groups of train tickets at the end of the questions:", memoriser_tree.named_steps["model"].get_n_leaves())

> **Check.** You should see `2210` groups. The tree asked questions until each group of train tickets had one answer.

> **Predict.** The rule "flag priority 1" catches 33% of the May escalations. How many of the 1,983 train escalations will the tree catch? How many of the 183 May escalations? Then run the cell.

In [ ]:
real = pd.DataFrame.from_dict({
    "memoriser on train": flag_report(train[TARGET], memoriser_tree.predict(train[FEATURES])),
    "memoriser on May": flag_report(valid[TARGET], memoriser_tree.predict(valid[FEATURES])),
    "rule on train": flag_report(train[TARGET], train["priority"] == 1),
    "rule on May": flag_report(valid[TARGET], valid["priority"] == 1),
}, orient="index")
real[["flagged", "caught", "missed", "precision", "recall"]]

> **Check.** You should see that the memoriser catches all `1983` train escalations, with precision and recall `1.0`. On May, it catches only `46` of 183 (recall `0.251`), with precision `0.26`. The rule has recall `0.349` on train and `0.333` on May.

On the tickets it learned from, the tree is perfect. On new tickets, it is worse than the simple rule. The rule changes little between train and May, because it learned a real pattern. The tree learned the noise of 16,939 old tickets.

> **Your turn.** From the table `real`, take the recall of the memoriser on May, and the recall of the memoriser on train. Put the **difference** (train minus May), rounded to 3 decimals, in `gap`. Run the cell, then the check cell.

In [ ]:
gap = ...  # recall on train minus recall on May, rounded to 3 decimals
print(gap)

In [ ]:
expect_hash('gap', gap, '3db68c49619f6833')

## 4. Why each look uses up the validation set

The validation set is for choosing: you try a change, look at the score on May, and keep the change if the score is better. But a choice made on 183 escalations is partly luck. The more versions you try, the more the winner's score is luck.

Tomás has 100 rules. Each one flags about 1 ticket in 5, chosen only from the ticket ID (`id_rule()` in the helper cell). The rules know nothing about the tickets, so each one should catch about 20% of the escalations.

> **Predict.** Tomás keeps the rule that catches the most May escalations. About what share does it catch on May? Then run the cell.

In [ ]:
may_recall = [flag_report(valid[TARGET], id_rule(number, valid))["recall"] for number in range(100)]
best = int(np.argmax(may_recall))
print("May, 100 rules: mean", round(np.mean(may_recall), 3), " lowest", min(may_recall), " highest", max(may_recall))
print("The best rule on May is rule", best)

> **Check.** You should see a mean of `0.199`, a lowest recall of `0.142` and a highest of `0.251`, for rule `7`.

Rule 7 looks clearly better than 20%. Is it? To learn its real catch rate, check it on many more tickets that played no part in the choice. The 16,939 train tickets are fine here: these rules learned nothing from them. Run the next cell.

In [ ]:
train_recall = [flag_report(train[TARGET], id_rule(number, train))["recall"] for number in range(100)]
print("Rule", best, "on train:", train_recall[best])
print("train, 100 rules: mean", round(np.mean(train_recall), 3), " lowest", min(train_recall), " highest", max(train_recall))

> **Check.** You should see `0.206` for rule 7 on train, and a mean of `0.2` for the 100 rules, from `0.177` to `0.229`.

Rule 7's 25.1% on May was luck. On train, it catches 20.6%, like every other rule. The chart below shows the 100 rules on both sets. Run it.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.scatter(np.array(may_recall) * 100, [1] * 100, alpha=0.5, label="May: 183 escalations")
ax.scatter(np.array(train_recall) * 100, [0] * 100, alpha=0.5, label="train: 1,983 escalations")
ax.scatter([may_recall[best] * 100, train_recall[best] * 100], [1, 0], s=120, facecolors="none", edgecolors="black", label=f"rule {best}")
ax.axvline(20, color="grey", linestyle="--")
ax.set_yticks([0, 1], ["train", "May"])
ax.set_xlabel("Escalations caught (%) by 100 rules that know nothing")
ax.set_ylim(-0.6, 1.6)
ax.legend(loc="center", ncol=3, fontsize=8)
plt.show()

> **Check.** You should see two rows of dots. The May row is wide (from about 14% to 25%). The train row is narrow, close to the dashed line at 20%. Rule 7 is on the right end of the May row and near the middle of the train row.

Two lessons follow:

- **A choice on the validation set picks some luck.** The more versions you compare on it, the more the winner's validation score is too good. So the validation score of your final choice is a little optimistic. That is why you keep a separate test set for the final score, and look at it once.
- **A small set moves more by chance.** With 183 escalations, a rule that knows nothing scores from 14% to 25%. With 1,983, from 18% to 23%. Section 6 uses this.

## 5. How to split: by time for Larkfield

A **random split** puts each row into train, validation or test by chance. It is fair when the rows are independent and their order does not matter. Larkfield's model must predict **future** tickets from past ones, so its split is by **time**, as in the SQL course: train is July 2024 to April 2026, validation is May 2026, and test is June 2026.

Run the next cell. It shows the escalation rate of the last six months.

In [ ]:
test = read("data/test.csv")
every = pd.concat([train, valid, test])
monthly = every.groupby(every["created_at"].dt.strftime("%Y-%m"))[TARGET].agg(["size", "mean"]).round(3)
monthly.tail(6)

> **Check.** You should see rates from `0.098` to `0.124` from January to April 2026, then `0.155` in May and `0.157` in June.

The new warranty policy started on 1 May 2026. No train month has a rate above 0.139. The time split shows the change: the validation set is the first look at the new conditions, as the model will meet them in real use. In this lesson, the test set is only counted, never used to judge or choose anything.

## 6. Cross-validation

May has only 183 escalations, so one score on it moves a lot by chance (section 4), and each look uses it up a little. **Cross-validation** gets more checks out of the 16,939 train tickets:

1. Cut the train tickets into 5 parts, the **folds**.
2. Fit the model on 4 folds, and score it on the fifth, which it did not see.
3. Repeat 5 times, so each fold is the "chapter exercises" once.

The 5 scores give a mean and a **standard deviation** (how much the score moves from fold to fold).

First, why the folds must be cut with care. Here are 20 rows: 5 escalated, at the top of the file. Run the cell.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

y_20 = np.array([1] * 5 + [0] * 15)
x_20 = np.arange(20).reshape(-1, 1)
for name, folds in [("KFold", KFold(5)), ("StratifiedKFold", StratifiedKFold(5))]:
    print(name, "escalated per fold:", [int(y_20[rows].sum()) for _, rows in folds.split(x_20, y_20)])

> **Check.** You should see `[4, 1, 0, 0, 0]` for `KFold` and `[1, 1, 1, 1, 1]` for `StratifiedKFold`.

Plain `KFold` cuts the rows in file order. Three of its folds have no escalation, so nothing can be caught in them. **Stratified** folds keep the share of escalations the same in every fold. The course always uses `StratifiedKFold(5, shuffle=True, random_state=0)`: stratified, shuffled, and the same folds on every run.

> **Predict.** In cross-validation, what share of escalations will the memorising tree catch on the 4 folds it learned from? And on the fold it did not see? Then run the cell. It fits the tree 5 times and takes a few seconds.

In [ ]:
from sklearn.model_selection import cross_validate

cv = StratifiedKFold(5, shuffle=True, random_state=0)
result = cross_validate(memorising_tree(), train[FEATURES], train[TARGET], cv=cv,
                        scoring="recall", return_train_score=True)
print("caught on the 4 folds it learned from:", result["train_score"].round(3))
print("caught on the fold it did not see:    ", result["test_score"].round(3))
print(f"mean {result['test_score'].mean():.3f}, standard deviation {result['test_score'].std():.3f}")

> **Check.** You should see `1.` five times, then `0.247 0.204 0.247 0.229 0.21`, with mean `0.227` and standard deviation `0.018`.

Note the names: scikit-learn calls the fold that the model did not see `test_score`. It is still part of our train set. The real test set (June) is not used.

Cross-validation found the memorising problem without May. Now the rule "flag priority 1", on the same folds. The rule learns nothing, so any change from fold to fold is chance. Run the cell.

In [ ]:
rule_recall = [flag_report(train[TARGET].iloc[rows], train["priority"].iloc[rows] == 1)["recall"]
               for _, rows in cv.split(train, train[TARGET])]
print("rule, caught per fold:", rule_recall)
print(f"mean {np.mean(rule_recall):.3f}, standard deviation {np.std(rule_recall):.3f}")

> **Check.** You should see `[0.371, 0.315, 0.343, 0.343, 0.376]`, mean `0.350` and standard deviation `0.022`.

The same rule scores from 31.5% to 37.6%, only because the folds hold different tickets. So a difference between two models that is smaller than this spread may be chance, not a real difference.

## 7. Your turn: 10 folds

Change one thing: the number of folds.

> **Predict.** With 10 folds, each fold has half as many tickets. Will the mean of the rule's recall change? Will the standard deviation get smaller or larger?

> **Your turn.** Make `cv_10 = StratifiedKFold(10, shuffle=True, random_state=0)`. Compute the rule's recall on each of its 10 folds, as in the cell above. Put the mean and the standard deviation, each rounded to 3 decimals, in the list `mean_and_std`. Run the cell, then the check cell.

In [ ]:
cv_10 = ...  # 10 stratified folds, shuffled, random_state=0
rule_recall_10 = ...  # the rule's recall on each fold
mean_and_std = ...  # [mean, standard deviation], each rounded to 3 decimals
print(mean_and_std)

In [ ]:
expect_hash('mean_and_std', mean_and_std, 'b7627ddaa88a1e79')

The mean stays the same: the rule is the same, and it is measured on the same tickets. The standard deviation is larger, because each fold has only about 198 escalations: a smaller set moves more by chance. The [lesson page](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#your-turn-10-folds) explains the trade-off.

## 8. Failure case: a random split hides the change

Tomás says: "A random split is the standard. Let's shuffle all the tickets and cut them." Run the cell. It makes a random split with the same sizes as the time split.

> **Predict.** What escalation rate will the random test set have: about 12% or about 16%? How many May and June tickets will be in the random train set?

In [ ]:
shuffled = every.sample(frac=1, random_state=0)
random_parts = {"test": shuffled.iloc[:1002], "valid": shuffled.iloc[1002:2181], "train": shuffled.iloc[2181:]}
for name, part in random_parts.items():
    recent = (part["created_at"] >= "2026-05-01").sum()
    print(f"random {name}: {len(part)} tickets, rate {part[TARGET].mean():.3f}, from May or June 2026: {recent}")

> **Check.** You should see a random test rate of `0.131` and `1947` May and June tickets in the random train set.

The random test set has a rate of 13.1%, mixed from all 24 months. The real future (June) has 15.7%. And the model would learn from 1,947 tickets of May and June, then be tested on older tickets: it would see the future during training. Nothing in the code warns you. The fix is the time split of the SQL course: train on the past, validate and test on the months after it.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/#knowledge-check). They count toward your certificate when you are signed in and enrolled.

In this lesson, you saw a model that memorises: it catches all 1,983 train escalations and only 46 of 183 in May, fewer than the rule "flag priority 1" (61). This is overfitting, and a score on the training data cannot show it. The training set is for learning, the validation set for comparing and choosing (each look uses it up a little), and the test set for one final score. Larkfield splits by time. Cross-validation scores a model 5 times on the train set, and its standard deviation shows how much a score moves by chance.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.